# P07 微调实战：Colab 免费 GPU + Qwen2.5-1.5B + LLaMA-Factory + QLoRA

> 目标：在 Colab 免费 T4 GPU 上，用 LLaMA-Factory 对 Qwen2.5-1.5B-Instruct 做 QLoRA 微调，得到「Java 面试专家」LoRA Adapter，再合并成完整模型并对比微调前后效果。
>
> 用法：菜单 Runtime → Change runtime type → GPU，然后 Run All。全部输出请复制保存贴回 WorkBuddy，用于撰写里程碑文档。

## 0. 验证云端 GPU

先确认 Colab 已分配 NVIDIA GPU。免费 GPU 型号不固定，能看到 `nvidia-smi` 输出即说明分配成功。

In [ ]:
!nvidia-smi

## 1. 安装 LLaMA-Factory

克隆官方仓库并以可编辑模式安装。安装完成后用 `llamafactory-cli version` 验证。

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e .
!llamafactory-cli version

## 2. 下载基座模型 Qwen2.5-1.5B-Instruct

用 `huggingface_hub.snapshot_download` 把基座模型拉到 `./models/qwen2.5-1.5b`，训练配置会引用这个本地路径，避免重复下载。

In [ ]:
!wget -q https://raw.githubusercontent.com/beiluoL/ai-engineer-journey/main/projects/07-open-source-llm/scripts/colab_download_model.py
!python colab_download_model.py

## 3. 准备 Java 面试数据集并注册

从本仓库拉取 `java_interview.json`（59 条真实 Java 面试题），写入 `data/` 并在 `data/dataset_info.json` 注册为 `java_interview`；同时写出训练与合并的 YAML 配置。

In [ ]:
!wget -q https://raw.githubusercontent.com/beiluoL/ai-engineer-journey/main/projects/07-open-source-llm/scripts/colab_prepare.py
!python colab_prepare.py

## 4. 开始 QLoRA 监督微调（SFT）

配置：LoRA rank=8 / alpha=16 / 4-bit 量化 / lr=2e-4 / epoch=3 / batch=1 / grad_accum=8。观察 loss 从约 2.x 逐步下降到 1 以下，得到 `saves/qwen2.5-1.5b/lora/java-interview` 这个 LoRA Adapter。

In [ ]:
!llamafactory-cli train examples/train_lora/qwen_qlora_java.yaml

## 5. 微调前对比（基座模型）

用三道训练集未覆盖的延伸问题，先记录基座模型的原始回答，作为对照基线。

In [ ]:
!wget -q https://raw.githubusercontent.com/beiluoL/ai-engineer-journey/main/projects/07-open-source-llm/scripts/colab_eval.py
!python colab_eval.py before

## 6. 合并 LoRA → 导出完整模型

把基座模型与 LoRA Adapter 合并，导出为独立可部署的 `models/qwen2.5-1.5b-java-interview-merged`。

In [ ]:
!llamafactory-cli export examples/merge_lora/qwen_java_export.yaml

## 7. 微调后对比（合并模型）

对同一组问题，对比合并后的回答，观察知识点覆盖、表达完整度、幻觉是否减少。

In [ ]:
!python colab_eval.py after

## 8. 收尾与下一步

你已完整走通：**数据集 → SFT → LoRA → QLoRA → GPU 训练 → Evaluation → Merge** 。

下一步：把 merged 模型下载到本地，用 Ollama / vLLM 部署，再由 Spring Boot + LangChain4j 调用，做成 Java AI 面试助手。

请把各 cell 的真实输出（loss 曲线、微调前后回答）复制保存并贴回 WorkBuddy 会话，我来渲染截图并撰写 P07 微调里程碑文档。